# 01 — Exploración de Datos

Carga del dataset, inspección de videos y del xlsx de anotaciones AVA 3.2.

In [1]:
import os, cv2, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH   = Path(os.environ.get("MICROCIRCULATION_DATA", "/content/drive/MyDrive"))
XLSX_PATH   = DATA_PATH / "grilla_trauma_sl.xlsx"
VIDEO_EXT   = ('.mp4', '.avi', '.mov')

video_files = sorted([f for f in DATA_PATH.iterdir() if f.suffix.lower() in VIDEO_EXT])
print(f"Videos encontrados: {len(video_files)}")
for v in video_files[:5]:
    print(" ", v.name)


ModuleNotFoundError: No module named 'google.colab'

## 1.1 Metadata de videos

In [ ]:
records = []
for vf in video_files:
    cap = cv2.VideoCapture(str(vf))
    if not cap.isOpened():
        continue
    records.append({
        "filename"  : vf.name,
        "width"     : int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height"    : int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "fps"       : cap.get(cv2.CAP_PROP_FPS),
        "frames"    : int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        "duration_s": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) / max(cap.get(cv2.CAP_PROP_FPS), 1)
    })
    cap.release()

meta_df = pd.DataFrame(records)
print(meta_df.describe())
meta_df.to_csv(DATA_PATH / "video_metadata.csv", index=False)


## 1.2 Cargar y limpiar xlsx AVA 3.2

In [ ]:
raw = pd.read_excel(XLSX_PATH, header=1)
raw.columns = (
    ['paciente', 'video', 'flow_0', 'flow_1', 'flow_2', 'flow_3', 'TOT',
     'small_vessel_density', 'TVD']
    + [f'V{i}' for i in range(1, 21)]
    + ['observaciones']
)
raw['paciente'] = raw['paciente'].fillna(method='ffill')

# Videos con datos válidos
df = raw[raw['TVD'].notna()].copy().reset_index(drop=True)

VEL_COLS = [f'V{i}' for i in range(1, 21)]

# Métricas derivadas
df['PPV']      = df['flow_3'] / df['TOT']
df['vel_mean'] = df[VEL_COLS].mean(axis=1)
df['HI']       = df[VEL_COLS].std(axis=1)          # Heterogeneity Index
df['n_vessels']= df[VEL_COLS].notna().sum(axis=1)  # vasos medidos

# Flag de calidad desde OBSERVACIONES
BAD = ['foco', 'inestable', 'back', 'compresión', 'compresion']
df['quality_ok'] = ~df['observaciones'].str.lower().str.contains(
    '|'.join(BAD), na=False)

print(f"Videos válidos: {len(df)}  |  Con buena calidad: {df['quality_ok'].sum()}")
df[['paciente','video','TVD','small_vessel_density','PPV','vel_mean','HI',
    'n_vessels','quality_ok']].head(10)


## 1.3 Distribución de métricas gold standard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
metrics = [('TVD','mm/mm²'), ('small_vessel_density','mm/mm²'),
           ('PPV','ratio'), ('vel_mean','µm/s'), ('HI','µm/s'), ('n_vessels','count')]

for ax, (col, unit) in zip(axes.flat, metrics):
    data_ok  = df.loc[df['quality_ok'], col].dropna()
    data_bad = df.loc[~df['quality_ok'], col].dropna()
    ax.hist(data_ok,  bins=15, alpha=0.7, label='Calidad OK',  color='steelblue')
    ax.hist(data_bad, bins=15, alpha=0.7, label='Con artefacto', color='tomato')
    ax.set_title(col); ax.set_xlabel(unit); ax.legend(fontsize=8)

plt.suptitle("Distribución de métricas AVA 3.2 (gold standard)", fontsize=14)
plt.tight_layout(); plt.savefig(DATA_PATH / "fig_01_distributions.png", dpi=150)
plt.show()


## 1.4 Primer frame de cada video

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, vf in zip(axes.flat, video_files[:12]):
    cap = cv2.VideoCapture(str(vf))
    ret, frame = cap.read(); cap.release()
    if ret:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(vf.name[:20], fontsize=8); ax.axis('off')
plt.tight_layout(); plt.savefig(DATA_PATH / "fig_01_frames.png", dpi=150)
plt.show()
print("Exploración completa. Continuar con NB02.")
